# Asignación de deliberación : método en la práctica

**Nivel:** 2  
**Requisitos previos:** Asignación de deliberación nivel 1  
**Tiempo de lectura:** 25 minutos

## Objetivos

El objetivo es explicar el problema y sus datos, inspeccionar y, en niveles superiores, ejecutar el método publicado en su tarea emparejada. Un planificador lento puede devolver una propuesta útil, una respuesta tardía, una salida inválida o no responder. El controlador debe considerar el resultado y la latencia del planificador mientras el entorno cambia; la incertidumbre no demuestra por sí sola que delegar ayude.

## Preparación

Las versiones siguientes están fijadas en `notebooks/requirements.txt` y también constan en los metadatos de este notebook.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Datos

El notebook lee artefactos comprometidos del producto. Propietario: Neuraxis. Repositorio: https://github.com/fsantibanezleal/CAOS_Neuraxis. Licencia: Apache-2.0. El JSON es un artefacto experimental propio y congelado; este notebook no redistribuye datos de benchmarks externos.

Este notebook lee `data/derived/index.json` y las trazas listadas. La ejecución usa un espacio aislado de caso Reflexmesh. Para métodos aprendidos, descarga los activos públicos exactos con `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000` y define `NEURAXIS_CHECKPOINTS` hacia esa carpeta. El manifiesto de publicación verifica los hashes. M09/M10 se leen desde trazas congeladas, salvo que `NEURAXIS_NOTEBOOK_MODEL` indique explícitamente un modelo Ollama local configurado.

Artefactos de esta lección:

- `data/derived/index.json`
- `data/derived/semantic-routing-study/analysis.json`
- `data/derived/episodes/M11-C13-nominal-40000.json`
- `data/derived/episodes/M12-C13-nominal-40000.json`
- `data/derived/episodes/M09-C13-nominal-40000.json`
- `data/derived/episodes/M10-C13-nominal-40000.json`

## Método

Inspecciona M09 planificación directa, M10 delegación asíncrona, M11 aplazamiento aprendido y M12 metacontrol como métodos separados. Recalcula los conteos observados del estudio de enrutamiento desde su informe persistido y compara sus brazos retenidos sin mezclarlos con el benchmark central.

Una estimación de valor de cómputo es \(VOC(c\mid x)=\mathbb E[V(x'\mid c,x)]-V_{act}(x)-\lambda_c cost(c)-\lambda_d delay(c)\). El valor esperado debe incluir planes fallidos, tardíos, inválidos y reemplazados. Esta es la formulación de investigación; el notebook no reajusta ni mejora en silencio la puerta histórica.

### Comprueba las versiones fijadas del entorno.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verifica la auditoría y los bytes de cada traza antes de interpretar resultados.

In [2]:
# Lee la evidencia comprometida y usa la API publicada; nunca define otra política.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M11', 'M12', 'M09', 'M10']
sample_case = 'C13'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C13 | seed: 40000
Methods: M11, M12, M09, M10


### Inspecciona las decisiones y resultados originales

In [3]:
# Resume la primera acción registrada, los candidatos autorizados y el resultado del oráculo independiente.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M11 | action= a-wait | reason= learned_incremental_benefit_gate | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 1 | steps= 2 | violations= 0
M12 | action= a-wait | reason= persistent_outcome_aware_metacontrol | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 1 | steps= 2 | violations= 0
M09 | action= a-ack_event | reason= Acknowledge the received event to confirm state processing and maintain responsiveness as per the goal. | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 0 | steps= 16 | violations= 0
M10 | action= a-wait | reason= asynchronous_fsm_planner | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 1 | steps= 2 | violations= 0


### Ejecuta las políticas publicadas y compara la corrida

In [4]:
# Los resultados de planificación se leen de las trazas congeladas; no hay llamada implícita a un LLM.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M11 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-wait'}
M12 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-wait'}
M09 not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above
M10 not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above


### Compara la secuencia completa y no solo la primera acción

In [5]:
# La igualdad de secuencia es diagnóstica; la verificación independiente decide el éxito.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M11 a-wait -> a-collect | terminal= succeeded | verified= 1
M12 a-wait -> a-collect | terminal= succeeded | verified= 1
M09 a-ack_event -> a-wait -> a-ack_event -> a-ack_event -> a-wait_dependency -> a-ack_event -> a-ack_event -> a-refresh -> a-refresh -> a-wait -> a-ack_event -> a-wait -> a-ack_event -> a-ack_event -> a-ack_event -> a-ack_event | terminal= failed | verified= 0
M10 a-wait -> a-collect | terminal= succeeded | verified= 1


## Resultados

El ajuste de la puerta usó 228 estados con siete etiquetas positivas y seleccionó el umbral 1,01, por lo que nunca delegó a través de esa puerta. El otro disparador de M12 es independiente. Informa llamadas al planificador, latencia y éxito verificado; configurar un LLM no demuestra que deliberar sea útil.

Este paso de autoría no compromete salidas generadas; `scripts/run_notebooks.py` las ejecuta y guarda desde un kernel nuevo.

## Resumen

La delegación debe justificar su costo con resultados emparejados. La evidencia actual informa un resultado negativo para la puerta aprendida y ninguna ventaja demostrada de M12 en éxito de tareas.

## Referencias

- [JEV](https://typesafe.ai/blog/introducing-system-one-models-and-jev)
- [DPT](https://aclanthology.org/2025.acl-long.206/)
- [VOC](https://arxiv.org/abs/1711.06892)
- [DEFER](https://proceedings.mlr.press/v119/mozannar20b.html)